##### Configuration and imports

In [ ]:
import ast
import uuid
import json
from delta.tables import DeltaTable
import re
import logging
import pyspark.sql.functions as F # import when, col, lit, to_date, to_timestamp, trim
from pyspark.sql import DataFrame
from datetime import date
import sys
import logging

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
logger = logging.getLogger("TransactionalSharedFunctions")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

##### setup_logging() function

In [ ]:
#1. Define a function for logging
def setup_logging():
    '''
    Set up logging configuration for Fabric notebooks.
    '''
    # Create formatter
    FORMAT = "%(asctime)s UTC - %(levelname)s - %(message)s (%(name)s)"
    formatter = logging.Formatter(fmt=FORMAT)

    # Get root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.WARNING)

    # Check if handlers exist, if not add console handler
    if not root_logger.handlers:
        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setFormatter(formatter)
        console_handler.setLevel(logging.INFO)
        root_logger.addHandler(console_handler)
    else:
        # Apply formatter to existing handlers
        for handler in root_logger.handlers:
            handler.setFormatter(formatter)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

##### Build source connection

In [ ]:
#1. Define a function that build a connection to db
def build_source_connection(source_type, source_location, source_connection_settings):
    # Only SQL database sources need a JDBC connection.
    if source_type != "sql_database":
        return None, None

    # Get the SQL server name from the connection settings.
    server_name = source_connection_settings.get("server_name")

    if not server_name:
        raise ValueError("server_name is required for sql_database sources")

    # Build the JDBC URL and authentication settings.
    jdbc_url = (
        "jdbc:sqlserver://"
        f"{server_name}:1433;"
        f"database={source_location};"
        "encrypt=true;"
    )

    connection_properties = {
        "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    }

    return jdbc_url, connection_properties





#2. Read one source table from SQL or lakehouse/shortcut.
def read_source_table(source_full_name, jdbc_url=None, connection_properties=None):
    if jdbc_url:
        source_query = f"(SELECT * FROM {source_full_name}) AS src"
        return spark.read.jdbc(jdbc_url, source_query, properties=connection_properties)

    return spark.table(source_full_name)

##### Ingestion data helpers

In [ ]:
#1. Convert a string(column Names) to snake_case. Purpose: Standardize column naming > framework source: nb_load_to_silver
def to_snake_case(string):
    string = re.sub(r'([a-z])([A-Z0-9])', r'\1_\2', string)
    string = re.sub(r'([A-Z])([A-Z][a-z])', r'\1_\2', string)
    string = string.lower().strip()
    return string





#2. Remove leading/trailing spaces from all string column values. input: Spark DataFrame > nb_load_to_silver: nb_load_to_silver
def trim_all_string_columns(df):
    string_columns = [field.name for field in df.schema.fields if field.dataType.typeName() == 'string']
    df = df.select(
        *[F.trim(F.col(c)).alias(c) if c in string_columns else F.col(c) for c in df.columns]
    )
    return df





#3. Replace old dates: If date ≤ 1900-01-01 → replace with 1900-01-01. Input:DataFrame > nb_load_to_silver: nb_load_to_silver
def replace_old_dates(df):
    date_columns = [field.name for field in df.schema.fields if field.dataType.typeName() in('date','timestamp')]
    for date_column in date_columns:
        df = df.withColumn(
            date_column, 
		    F.when(
			    F.col(date_column) <= F.to_date(F.lit('1900-01-01')), F.to_date(F.lit('1900-01-01'), 'yyyy-MM-dd')
            )
			.otherwise(
                F.col(date_column)
            )
        )
    return df





#4. Define parent function that applies all cleaning steps to the DataFrame. > nb_load_to_silver: nb_load_to_silver
def clean_bronze_table(df, limit_rows_for_debugging=False):
    if limit_rows_for_debugging:
        df = df.limit(3)
    snake_case_column_names = [to_snake_case(col) for col in df.columns]
    df = df.toDF(*snake_case_column_names)
    df = replace_old_dates(df)
    df = trim_all_string_columns(df)
    return df





#5. Add metadata columns to dataframe > nb_load_to_silver: nb_load_to_delta_table
def add_metadata(df: DataFrame, ingest_date: str, file_path: str, schema_name: str, table_name: str, lineage_id: str) -> DataFrame:
    return (
        df
        .withColumn("_ingest_time", F.current_timestamp()) # cast to hh:mm:00
        .withColumn("_ingest_date", F.lit(ingest_date))
        .withColumn("_source_file", F.lit(file_path)) #source_table
        .withColumn("_source_system", F.lit(schema_name))
        .withColumn("_table", F.lit(table_name)) #target_table
        .withColumn("_lineage_id", F.lit(lineage_id))
    )

def add_metadata_v2(df, source_system, source_table, target_table, lineage_id):
    return (
        df
        .withColumn("_ingest_timestamp", F.current_timestamp())
        .withColumn("_source_system", F.lit(source_system))
        .withColumn("_source_table", F.lit(source_table))
        .withColumn("_target_table", F.lit(target_table))
        .withColumn("_lineage_id", F.lit(lineage_id))
    )





#6. Removes duplicates from dataframe > new function
def remove_exact_duplicates(df):
    return df.dropDuplicates(df.columns)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

##### Load data helpers

In [ ]:
#1. Adds hash columns used by merge-scd1 loads.
def add_hash_columns(df, target_pk):
    if not target_pk:
        raise ValueError("target_pk is required to add hash columns")

    # Build one hash from the target primary key columns.
    df = df.withColumn("_hashed_pk", F.md5(F.concat_ws("|", *[F.coalesce(F.col(col).cast("string"), F.lit("<NULL>")) for col in target_pk])))

    # Build one hash from the non-key business columns.
    row_hash_cols = col for col in df.columns if col not in target_pk + ["_hashed_pk"]

    df = df.withColumn("_hashed_row", F.md5(F.concat_ws("|", *[F.coalesce(F.col(col).cast("string"), F.lit("<NULL>")) for col in row_hash_cols])))

    return df





#2. Check whether a business key has duplicates and display samples. > new function
def validate_duplicates(df, key_col="_hashed_pk", limit=5):
    # Unpacks primary kays or takes _hashed_pk
    group_cols = key_col if isinstance(key_col, list) else [key_col]

    # groups by primary key from table
    dup_keys_df = (
        df.groupBy(*group_cols)
          .agg(F.count("*").alias("duplicate_count"))
          .filter(F.col("duplicate_count") > 1)
          .orderBy(F.col("duplicate_count").desc())
    )

    # print # duplicate rows
    total_dup_groups = dup_keys_df.count()
    print(f"Total duplicate {group_cols} groups: {total_dup_groups}")

    # if there are duplicates, return full dataframe
    if total_dup_groups > 0:
        dup_records_df = (
            df.join(dup_keys_df.select(*group_cols), on=group_cols, how="inner")
              .orderBy(*group_cols)
        )
        display(dup_records_df.limit(limit))





#3. Framework-aligned target load helper. >  nb_load_to_silver: nb_load_to_silver
def load_to_target(df, target_full_name, target_load_strategy, target_location_path=""):
    # Get schema and table name from target_table_path
    target_schema, target_table = target_full_name.split(".")

    # Set load metrics
    rows_inserted = 0
    rows_updated = 0
    rows_read = 0
    rows_copied = 0

    # If target path does not exist, create delta files and table
    if not DeltaTable.isDeltaTable(spark, target_location_path):
        # Create schema
        logger.info(f'Creating new schema {target_schema}')
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")
        
        # Create delta files 
        logger.info(f'Creating new delta files at {target_location_path}')
        df.write.format("delta").mode("overwrite").save(target_location_path)

        # Create delta table 
        logger.info(f'Creating new delta table {target_full_name}')
        spark.sql(f"CREATE TABLE IF NOT EXISTS {target_full_name} USING DELTA LOCATION '{target_location_path}'")

        # Update row metrics
        rows_inserted = df.count()
        rows_read = df.count()
        rows_copied = df.count()

    else:
        match target_load_strategy:
            
            case "merge-scd1":
                # Merge inserted and changed rows
                logger.info(f"Running merge-scd1 into {target_full_name}")
                
                # Look for existing delta files
                delta_table = DeltaTable.forPath(spark, target_location_path)
                df_target = spark.read.format("delta").load(target_location_path)

                # Update row metrics
                rows_read = df.count()
                rows_inserted = df.alias("s").join(df_target.alias("t"), "_hashed_pk", "left_anti").count()
                rows_updated = df.alias("s").join(df_target.alias("t"), "_hashed_pk", "inner").filter("NOT (t._hashed_row <=> s._hashed_row)").count()
                rows_copied = rows_inserted + rows_updated

                if rows_copied == 0:
                    logger.info(f"Skipping {target_full_name}: no changes")

                # Run merge-scd1
                else:
                    (
                        delta_table.alias("t")
                        .merge(df.alias("s"), "t._hashed_pk = s._hashed_pk")
                        .whenMatchedUpdateAll(condition="NOT (t._hashed_row <=> s._hashed_row)")
                        .whenNotMatchedInsertAll()
                        .execute()
                    )


            case "overwrite":
                # Replace all target data
                logger.info(f"Overwriting records in {target_full_name}")
                
                # Update row metrics
                rows_read = df.count()
                rows_copied = rows_read

                # Run overwrite
                df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(target_location_path)

            
            case "append":
                # Append all source rows
                logger.info(f"Appending new records to {target_full_name}")
                
                # Update row metrics
                rows_read = df.count()
                rows_inserted = rows_read
                rows_copied = rows_read

                # Run append
                df.write.mode("append").format("delta").save(target_location_path)


            case _:
                raise ValueError(f"Unsupported target_load_strategy: {target_load_strategy}")

    # Return load metrics
    return {
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_read": rows_read,
        "rows_copied": rows_copied
    }

StatementMeta(, , -1, Cancelled, , Cancelled, True)